In [1]:
# Calculate the PM2.5 Mortality per health variable

In [2]:
import os
import xarray as xr
import numpy as np
from utils.utils import get_scenario_config

In [3]:
# === Health variables ===
# COPD (chronic obstructive pulonary disease)
# LRI (lower respiratory infection)
# IHD (ischemic heart disease)
# DM2 (type 2 diabetes)
# LC (tracheal, bronchus, and lung cancer)
# Stroke
health_vars = ["DM", "LC", "COPD", "LRI"]
age_health_vars = ["Stroke", "IHD"]

In [4]:
# === Path config ===
BMR_DIR = "/glade/work/awells/air_quality/BMR/"

In [5]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "G6-1.5K"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

RR_DIR = f"/glade/work/awells/air_quality/{model}/pm25/RR/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/mortality/pm25/"

n_samples = 1000

for health_VAR in health_vars:
    for ens_num in ensemble_members:
        print(f"Processing {scenario} ensemble member {ens_num:02d} for {health_VAR}")

        dates = f"{years.start}-{years.stop}"

        RR_file = f"RR_{model}_{health_VAR}_{scenario}_{ens_num:02d}_{dates}.nc"
        RR_path = os.path.join(RR_DIR, RR_file)
        RR = xr.open_dataarray(RR_path)

        PAF = 1 - (1/RR)

        bmr_file = f"GBD_BMR_Country_{health_VAR}_newlabels_1990-2009.nc"
        bmr_path = os.path.join(BMR_DIR, bmr_file)
        bmr = xr.open_dataarray(bmr_path)

        # Extract mean, lower, upper
        BMR_mean = bmr.sel(quantile="mean")
        BMR_lower = bmr.sel(quantile="lower")
        BMR_upper = bmr.sel(quantile="upper")

        # Estimate standard deviation assuming 95% CI
        BMR_std = (BMR_upper - BMR_lower) / (2 * 1.96)

        # Monte Carlo sampling (shape: [country, sample])
        BMR_samples = np.random.normal(
            loc=BMR_mean.values[..., np.newaxis],
            scale=BMR_std.values[..., np.newaxis],
            size=(len(bmr.country), n_samples)
        )

        # Convert to xarray
        BMR_samples = xr.DataArray(
            BMR_samples,
            dims=("country", "sample"),
            coords={
                "country": bmr.country,
                "sample": np.arange(n_samples)
            }
        )

        mortality = BMR_samples * PAF

        description = (f"PM2.5 Attributable Mortality for {health_VAR} "
                       "- scripts by A.F. Wells (2025)")

        mortality.attrs["description"] = description
        mortality.attrs["ensemble"] = ens_num
        mortality.attrs["scenario"] = scenario

        out_file = f"PM2.5_Mortality_{health_VAR}_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)
        mortality.to_netcdf(out_path)

print("All processing complete.")

Processing G6-1.5K ensemble member 01 for DM
Processing G6-1.5K ensemble member 02 for DM
Processing G6-1.5K ensemble member 03 for DM
Processing G6-1.5K ensemble member 01 for LC
Processing G6-1.5K ensemble member 02 for LC
Processing G6-1.5K ensemble member 03 for LC
Processing G6-1.5K ensemble member 01 for COPD
Processing G6-1.5K ensemble member 02 for COPD
Processing G6-1.5K ensemble member 03 for COPD
Processing G6-1.5K ensemble member 01 for LRI
Processing G6-1.5K ensemble member 02 for LRI
Processing G6-1.5K ensemble member 03 for LRI
All processing complete.
